# Gene essentiality on a kidney functional network

## Why this run

We replicated HELP first. Most of its predictive signal came from the CCcfs features,
which are subcellular localization annotations from COMPARTMENTS. Not from the network.
HELP's own feature importance analysis says the same thing.

In HELP the PPI enters as node2vec embeddings. What it gives the model is connectivity.
Essential genes tend to be well connected, so connectivity helps, but the actual
prediction comes from the other features.

Here the graph is the thing we want to study, not a connectivity prior. So a functional
network is a better fit than a physical one. Its edges say two genes work together,
rather than that their proteins touch.

This run swaps the IID physical PPI for the HumanBase kidney functional network.
Nothing else changes.

## Setup

- Graph: HumanBase kidney functional network, posterior >= 0.2. Nodes are genes, edges
  are functional association. One graph per cell line, same topology in all of them.
- Node features: expression (log1p TPM) and damaging mutation, per gene per cell line.
  These two numbers are the only thing that differs between the 32 graphs.
- Labels: DepMap 25Q3 Chronos gene effect, 32 kidney cell lines.
- Model: 2 layer GATv1, 4 heads, edge posterior as an edge feature.
- Evaluation: 5 fold, split by cell line. Genes are never split, so no gene is in both
  train and test.

## What to look at

There are two outputs. The Pearson r says how well essentiality was predicted. The
attention weights say which neighbours the model used to predict it.

The attention weights are the part worth reading. They are a learned weighting over
gene pairs, so they can be checked against known complexes.


In [ ]:
!pip install torch-geometric


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
!pip install torch-geometric



In [4]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.stats import pearsonr
from scipy.stats import pearsonr

import random
from torch_geometric.loader import DataLoader


In [5]:
# HumanBase kidney functional network, posterior >= 0.2
ppi = pd.read_csv("/content/humanbase_edges.csv")
labels = pd.read_csv("/content/labels.csv")
expressions = pd.read_csv("/content/expression.csv")
mutations = pd.read_csv("/content/mutations.csv")

In [ ]:
ppi.shape

In [7]:
# node index comes from the network, not from DepMap
ppi_genes = pd.concat([ppi['symbol_a'], ppi['symbol_b']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}

In [8]:
edge_src = ppi['symbol_a'].map(gene_to_idx).values
edge_dst = ppi['symbol_b'].map(gene_to_idx).values

# symmetrised, so messages pass both ways
edge_index = torch.tensor(
    np.array([np.concatenate([edge_src, edge_dst]),
              np.concatenate([edge_dst, edge_src])]),
    dtype=torch.long
)


# posterior, raw. duplicated to match the doubled edge_index
edge_features = torch.tensor(ppi[['posterior']].values, dtype=torch.float)
edge_attr = torch.cat([edge_features, edge_features], dim=0)

In [9]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
# genes missing from the mutation file become 0
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)

In [10]:
# one graph per cell line. topology and edge_attr are shared, only x differs
graph_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)

    y_raw = labels_reindexed.iloc[i]
    # unlabelled genes filled with 0 for shape, then masked out of the loss
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)

In [11]:
import torch.nn as nn
from torch_geometric.nn import GATConv

In [12]:
class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        # one encoder per feature, 32 + 32 -> 64
        self.expr_encoder = nn.Linear(1, 32)
        self.mut_encoder = nn.Linear(1, 32)
        # no fill_value, so PyG fills self-loops with the mean posterior
        self.conv1 = GATConv(64, 64, heads=4, edge_dim=1)
        # heads concat in layer 1 (64*4 in), average in layer 2
        self.conv2 = GATConv(64 * 4, 64, heads=4, edge_dim=1, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        expr = self.expr_encoder(x[:, 0:1])
        mut = self.mut_encoder(x[:, 1:2])
        x = torch.cat([expr, mut], dim=1)

        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        # one score per gene
        x = self.out(x)
        return x.squeeze()

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [ ]:
from sklearn.model_selection import KFold

# folds split cell lines, not genes
kf = KFold(n_splits=5, shuffle=True, random_state=12)
all_attn = []
fold_pearsons = []
all_curves = []

for fold, (train_idx, test_idx) in enumerate(kf.split(graph_list)):
    train_data = [graph_list[i] for i in train_idx]
    test_data = [graph_list[i] for i in test_idx]
    train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=1)

    # fresh model per fold
    model = KidneyGAT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.MSELoss()

    train_curve, test_curve = [], []

    for epoch in range(200):
        model.train()
        ep_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        train_curve.append(ep_loss / len(train_loader))

        # test loss per epoch, for convergence. iterating a loader advances
        # torch's RNG and dropout draws from it, so save and restore around
        # this block to leave training unchanged.
        _cpu_rng = torch.random.get_rng_state()
        _cuda_rng = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        model.eval()
        with torch.no_grad():
            t_loss = 0
            for batch in test_loader:
                batch = batch.to(device)
                pred = model(batch)
                t_loss += loss_fn(pred[batch.mask], batch.y[batch.mask]).item()
            test_curve.append(t_loss / len(test_loader))
        torch.random.set_rng_state(_cpu_rng)
        if _cuda_rng is not None:
            torch.cuda.set_rng_state_all(_cuda_rng)

        if epoch % 20 == 0:
            print(f"  ep {epoch:3d}  train {train_curve[-1]:.4f}  test {test_curve[-1]:.4f}")

    all_curves.append((train_curve, test_curve))

    model.eval()
    fold_corrs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            pred = model(batch)
            # correlation across genes, within one held-out cell line
            p = pred[batch.mask].cpu().numpy()
            a = batch.y[batch.mask].cpu().numpy()
            r, _ = pearsonr(p, a)
            fold_corrs.append(r)

            # forward pass repeated to reach the attention weights.
            # conv1's are discarded, only conv2's are kept.
            expr = model.expr_encoder(batch.x[:, 0:1])
            mut = model.mut_encoder(batch.x[:, 1:2])
            x = torch.cat([expr, mut], dim=1)
            x, _ = model.conv1(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)
            x = model.relu(x)
            x = model.dropout(x)
            x, (edge_idx, scores) = model.conv2(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)
            # average over the 4 heads
            avg_scores = scores.mean(dim=1).cpu().numpy()
            all_attn.append(avg_scores)

    fold_r = np.mean(fold_corrs)
    fold_pearsons.append(fold_r)
    print(f"Fold {fold+1}: Pearson r = {fold_r:.4f}")

print(f"\nMean Pearson r across 5 folds: {np.mean(fold_pearsons):.4f} ± {np.std(fold_pearsons):.4f}")

In [ ]:
# Get edge indices from one sample
# this pass is only for edge_idx, identical across graphs. `model` here is
# the last fold's.
sample = graph_list[0].to(device)
with torch.no_grad():
    expr = model.expr_encoder(sample.x[:, 0:1])
    mut = model.mut_encoder(sample.x[:, 1:2])
    x = torch.cat([expr, mut], dim=1)
    x, _ = model.conv1(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)
    x = model.relu(x)
    x = model.dropout(x)
    x, (edge_idx, _) = model.conv2(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)

src = edge_idx[0].cpu().numpy()
dst = edge_idx[1].cpu().numpy()

# Average attention across all folds/cell lines
# 32 arrays, one per held-out cell line
mean_attn = np.mean(all_attn, axis=0)

# Filter self-loops
# PyG appended one per node
mask = src != dst
src_filtered = src[mask]
dst_filtered = dst[mask]
attn_filtered = mean_attn[mask]

idx_to_gene = {i: gene for gene, i in gene_to_idx.items()}

# Top 100 interactions
# unconditional ranking, not degree-normalised
top_indices = np.argsort(attn_filtered)[-100:][::-1]
print("Top 100 gene interactions (averaged across all folds):\n")
for rank, i in enumerate(top_indices, 1):
    g1 = idx_to_gene[src_filtered[i]]
    g2 = idx_to_gene[dst_filtered[i]]
    print(f"{rank:2d}. {g1:10s} <-> {g2:10s}  attn: {attn_filtered[i]:.4f}")

In [ ]:
# per-gene view: conditioned on one gene, where does its attention go
known_cancer_genes = ['VHL', 'TP53', 'EGFR', 'PTEN', 'MTOR', 'MYC', 'KRAS', 'BRCA1', 'PIK3CA', 'BRAF']

for gene in known_cancer_genes:
    if gene not in gene_to_idx:
        print(f"{gene} — not in PPI network")
        continue

    idx = gene_to_idx[gene]
    # either direction, since the graph was symmetrised
    gene_mask = (src_filtered == idx) | (dst_filtered == idx)
    gene_attn = attn_filtered[gene_mask]
    gene_src = src_filtered[gene_mask]
    gene_dst = dst_filtered[gene_mask]

    if len(gene_attn) == 0:
        print(f"{gene} — no edges found")
        continue

    top5 = np.argsort(gene_attn)[-5:][::-1]

    print(f"\n{gene} — {len(gene_attn)} edges, max attn: {gene_attn.max():.4f}")
    for i in top5:
        partner = idx_to_gene[gene_dst[i]] if gene_src[i] == idx else idx_to_gene[gene_src[i]]
        print(f"  <-> {partner:12s}  attn: {gene_attn[i]:.4f}")